# Синтетика «Большой Сочи» для агента «О!дин запрос»

Каталог для инструментов агента: районы, климат, отели (номера, тарифы, отзывы), POI, расписания самолётов/поездов/автобусов, местные линии, тарифы трансфера, правила цен и крайние случаи.

**Принципы**
- Python задаёт структуру, квоты и все числа; GLM-5.3 пишет тексты (названия, описания, номера, отзывы, POI).
- Реалистичность не цель, цель — согласованность: время в пути выводится из км (`synth/geo.py`), цены из формул.
- Офферы на даты не хранятся: их считает `synth/pricing.py::Catalog` детерминированно, выход — модели `harness/contracts.py`.
- Ответы GLM кэшируются в `synth/raw/cache/`: повторный прогон идёт без сети. `OFFLINE = True` запрещает сетевые вызовы.

Запуск из корня репо: `uv run --with jupyter jupyter lab synth/synth.ipynb`.

In [1]:
import asyncio
import json
import logging
import sys
from collections import Counter
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "harness").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd

from synth import generate as G
from synth import refdata as R
from synth import validate as V
from synth.llm import GLM
from synth.pricing import Catalog

logging.basicConfig(level=logging.WARNING)
OUT = ROOT / "synth" / "data"
OUT.mkdir(parents=True, exist_ok=True)
OFFLINE = False
glm = GLM(model="glm-5.3-no-collect", concurrency=8, offline=OFFLINE)

def dump(name, obj):
    def conv(x):
        return x.model_dump(mode="json") if hasattr(x, "model_dump") else x
    data = [conv(x) for x in obj] if isinstance(obj, list) else (
        {k: [conv(i) for i in v] if isinstance(v, list) else conv(v) for k, v in obj.items()} if isinstance(obj, dict) else conv(obj))
    (OUT / name).write_text(json.dumps(data, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"{name}: {len(obj) if hasattr(obj, '__len__') else 1}")

## 1. Справочники в коде: климат, точки прибытия, тарифы трансфера, правила цен

In [2]:
climate = R.climate_rows()
dump("climate.json", climate)
dump("reference_points.json", R.REFERENCE_POINTS)
dump("transfer_tariffs.json", R.TRANSFER_TARIFFS)
dump("pricing_rules.json", R.PRICING)
pd.DataFrame([c.model_dump() for c in climate]).pivot(index="month", columns="zone", values=["air_day_c", "sea_c"])

climate.json: 36
reference_points.json: 10
transfer_tariffs.json: 6
pricing_rules.json: 19


air_day_c                      sea_c                     
zone      coast imeretinka mountains coast imeretinka mountains
month                                                          
1          11.0       11.0       3.0  10.0       10.0       NaN
2          11.0       11.0       4.0   9.0        9.0       NaN
3          13.0       13.0       8.0  10.0       10.0       NaN
4          17.0       17.0      14.0  12.0       12.0       NaN
5          21.0       21.0      19.0  17.0       17.0       NaN
6          25.0       25.0      23.0  22.0       22.0       NaN
7          28.0       28.0      26.0  25.0       25.0       NaN
8          29.0       29.0      26.0  26.0       26.0       NaN
9          25.0       25.0      22.0  24.0       24.0       NaN
10         21.0       21.0      16.0  20.0       20.0       NaN
11         16.0       16.0      10.0  16.0       16.0       NaN
12         13.0       13.0       5.0  13.0       13.0       NaN

## 2. Районы (скелет в коде, тексты — GLM) и перевозчики

In [3]:
areas = await G.gen_areas(glm)
carriers = await G.gen_carriers(glm)
area_by_id = {a.area_id: a for a in areas}
print(carriers)
pd.DataFrame([{"id": a.area_id, "zone": a.zone, "price": a.price_level, "best_for": a.best_for, "vibe": a.vibe} for a in areas])

airlines=['Аврора Лайн', 'СкайЮг', 'Кубань Эйрвейз', 'Меридиан Авиа', 'ЮгТранс Авиалинии', 'СоколЭйр'] rail=['КурортЭкспресс', 'Южный Путь', 'Магистраль Дон Трэвел'] local=['СочиЭлектро', 'Черноморские маршруты']


,id,zone,price,best_for,vibe
0,lazarevskoe,coast,low,"[море, бюджетно, с детьми, тусовка]",Шумный приморский городок с пляжем в двух шага...
1,loo,coast,low,"[море, тихо, бюджетно]","Задушевная тишина у моря, где отдыхают дачники..."
2,dagomys,coast,medium,"[с детьми, оздоровление, море]",Старая санаторная классика среди субтропически...
3,sochi_center,coast,high,"[тусовка, экскурсии, море, романтика]","Большой город у моря, который не спит до утра"
4,matsesta,coast,medium,"[оздоровление, тихо, море]",Тихие процедуры и запах сероводорода вместо кл...
5,khosta,coast,low,"[бюджетно, тихо, море, экскурсии]",Дикая природа в двадцати минутах от городских ...
6,adler,coast,medium,"[с детьми, море, активный отдых]",Семейный курорт с аквапарком за окном и аэропо...
7,imeretinka,imeretinka,high,"[с детьми, активный отдых, море, люкс]","Олимпийская новизна, простор и велосипеды вдол..."
8,krasnaya_polyana,mountains,medium,"[горы/лыжи, бюджетно, активный отдых]",Горная деревня с мангальным дымом и видом на с...
9,rosa_khutor,mountains,premium,"[горы/лыжи, люкс, активный отдых, романтика]",Русские Альпы с шампанским на террасе


## 3. Отели
Слоты задают тип размещения, звёзды, линию от моря / расстояние до подъёмника, животных, adults-only, семейный номер и варианты питания — с гарантиями покрытия (≥2 pet-friendly и номер 2+2 в каждом районе). GLM заполняет название, адрес, описание, удобства, теги и номера.

In [4]:
slots = {a["area_id"]: G.build_slots(a) for a in R.AREAS}
pd.DataFrame([{**s, "area": k} for k, v in slots.items() for s in v]).pivot_table(
    index="area", columns="property_type", values="slot", aggfunc="count", fill_value=0)

property_type,apartments,glamping,guest_house,hotel,resort,sanatorium
area,,,,,,
adler,3,0,6,9,2,2
dagomys,2,0,4,6,1,1
imeretinka,4,0,2,9,3,0
khosta,2,0,4,6,1,1
krasnaya_polyana,3,2,2,7,2,0
lazarevskoe,3,0,5,7,1,2
loo,2,0,4,6,1,1
matsesta,1,0,3,5,1,4
rosa_khutor,3,2,2,6,1,0


In [5]:
cards = await asyncio.gather(*(G.gen_hotels(glm, area_by_id[a], slots[a]) for a in slots))
hotels = []
for a, cs in zip(slots, cards):
    for i, (s, c) in enumerate(zip(slots[a], cs)):
        hotels.append(G.finalize_hotel(area_by_id[a], s, c, i))
G.dedupe_names(hotels, area_by_id)
print(len(hotels), "объектов,", sum(len(h.rate_plans) for h in hotels), "тарифов,", glm.stats)

166 объектов, 1384 тарифов, {'calls': 0, 'cache_hits': 34, 'retries': 0, 'tokens_out': 0}


## 4. Отзывы (второй проход GLM, вход — готовая карточка отеля) и POI

In [6]:
reviews = await asyncio.gather(*(G.gen_reviews(glm, h, area_by_id[h.area_id]) for h in hotels))
reviews = G.inject_reviews([r for rs in reviews for r in rs], hotels)
G.aggregate_aspects(hotels, reviews)
poi = [p for ps in await asyncio.gather(*(G.gen_poi(glm, a) for a in areas)) for p in ps]
print(len(reviews), "отзывов,", sum(r.is_injection for r in reviews), "с инъекцией;", len(poi), "POI;", glm.stats)

1010 отзывов, 5 с инъекцией; 40 POI; {'calls': 0, 'cache_hits': 210, 'retries': 0, 'tokens_out': 0}


## 5. Транспорт, расстояния, крайние случаи

In [7]:
flights = G.build_flights(carriers)
trains = G.build_trains(carriers, {a.area_id: a.model_dump() for a in areas})
edge = G.build_edge_cases(hotels, flights, areas)
print(len(flights), "рейсов,", Counter(t.kind for t in trains))
pd.DataFrame(G.area_distances(areas)).query("point_id == 'aer'")[["area_id", "km", "drive_min"]]

82 рейсов, Counter({'train': 30, 'bus': 4, 'local_train': 2})


,area_id,km,drive_min
0,lazarevskoe,100.0,165
10,loo,54.0,95
20,dagomys,45.0,80
30,sochi_center,30.0,60
40,matsesta,23.0,50
50,khosta,16.0,40
60,adler,4.0,20
70,imeretinka,11.0,30
80,krasnaya_polyana,44.0,65
90,rosa_khutor,50.0,70


## 6. Экспорт

In [8]:
dump("areas.json", areas)
dump("hotels.json", hotels)
(OUT / "reviews.jsonl").write_text("\n".join(r.model_dump_json() for r in reviews) + "\n", encoding="utf-8")
dump("poi.json", poi)
dump("carriers.json", carriers)
dump("transport_schedules.json", {"origins": R.ORIGINS, "flights": flights, "trains": trains, "local_lines": R.LOCAL_LINES})
dump("area_distances.json", G.area_distances(areas))
dump("edge_cases.json", edge)
dump("meta.json", G.catalog_meta())

areas.json: 10
hotels.json: 166
poi.json: 40
carriers.json: 1
transport_schedules.json: 4
area_distances.json: 100
edge_cases.json: 4
meta.json: 10


## 7. Проверки: согласованность, покрытие, детерминизм, сборка поездки через `harness.checks`

In [9]:
cat = Catalog(OUT)
results = V.run_all(cat)
for k, v in results.items():
    print(f"{k}: {'OK' if not v else v}")
assert not any(results.values()), results

consistency: OK
coverage: OK
determinism: OK
aliases: OK
example_itinerary: OK


In [10]:
it, info = V.example_itinerary(cat)
info

{'hotel': 'Гостевой дом «У Тамары и Мити»',
 'hotel_total': 30516.0,
 'out': 'Кубань Эйрвейз КЭ-708, без багажа 11:20→13:50',
 'back': 'Кубань Эйрвейз КЭ-373, без багажа 15:30',
 'transfer_in': ('taxi', 15, 1410.0),
 'total': 73626,
 'checks': [('arrival_vs_checkin',
   'info',
   'Прибытие в отель ≈14:50, заселение с 14:00.'),
  ('checkout_vs_departure', 'info', 'Выезд 12:00, к транспорту — к 13:15.')]}

## 8. Распределения

In [11]:
hdf = pd.DataFrame([{**h.model_dump(exclude={"room_types", "rate_plans", "review_aspects", "description"}),
                     "rooms": len(h.room_types), "rates": len(h.rate_plans)} for h in hotels])
display(hdf.groupby("property_type")[["base_price_per_night", "rating", "rooms"]].describe().round(0))
display(pd.crosstab(hdf.area_id, hdf.stars.fillna(0).astype(int)))
display(pd.Series(Counter(a for h in hotels for a in h.amenities)).sort_values(ascending=False).head(20))
pd.DataFrame([r.model_dump() for r in reviews]).groupby("traveler_type").score.describe()

base_price_per_night                                             \
                             count     mean     std     min      25%      50%   
property_type                                                                   
apartments                    26.0   4450.0  1305.0  2800.0   3500.0   4200.0   
glamping                       4.0   9275.0  2060.0  7500.0   7500.0   9150.0   
guest_house                   38.0   2671.0   753.0  1900.0   2100.0   2500.0   
hotel                         70.0   8741.0  6201.0  2000.0   4025.0   7050.0   
resort                        15.0  18240.0  5205.0  8800.0  15350.0  17000.0   
sanatorium                    13.0   6546.0  1491.0  3800.0   6100.0   6600.0   

                                rating       ...            rooms            \
                   75%      max  count mean  ...  75%   max count mean  std   
property_type                                ...                              
apartments      5275.0   7800.0   26.0  8.0  ...  8.0   9.0  26.0  3.0  0.0   
glamping       10925.0  11300.0    4.0  8.0  ...  8.0   9.0   4.0  3.0  0.0   
guest_house     3100.0   4800.0   38.0  8.0  ...  8.0   9.0  38.0  2.0  1.0   
hotel          11150.0  28200.0   70.0  8.0  ...  8.0  10.0  70.0  3.0  0.0   
resort         22600.0  26300.0   15.0  8.0  ...  8.0   9.0  15.0  3.0  0.0   
sanatorium      7200.0   8900.0   13.0  8.0  ...  8.0   9.0  13.0  3.0  0.0   

                                        
               min  25%  50%  75%  max  
property_type                           
apartments     2.0  2.0  3.0  3.0  3.0  
glamping       3.0  3.0  3.0  3.0  3.0  
guest_house    2.0  2.0  2.0  3.0  3.0  
hotel          2.0  3.0  3.0  3.0  4.0  
resort         3.0  3.0  3.0  3.0  4.0  
sanatorium     3.0  3.0  3.0  3.0  3.0  

[6 rows x 24 columns]

stars,0,2,3,4,5
area_id,,,,,
adler,9,0,5,5,3
dagomys,6,0,3,2,3
imeretinka,6,0,0,5,7
khosta,6,0,4,3,1
krasnaya_polyana,7,0,1,5,3
lazarevskoe,8,2,5,3,0
loo,6,2,3,2,1
matsesta,4,0,6,3,1
rosa_khutor,7,0,0,4,3


wifi                166
parking             147
air_conditioning    127
restaurant          104
laundry             103
balcony              95
elevator             89
bar                  87
room_service         72
garden               72
sea_view             70
gym                  65
kitchen              58
spa                  57
mountain_view        45
bbq                  44
pool                 42
sauna                41
playground           38
beach_shuttle        29
dtype: int64

,count,mean,std,min,25%,50%,75%,max
traveler_type,,,,,,,,
business,107.0,7.635514,1.102249,5.0,7.0,8.0,8.00,10.0
couple,234.0,8.277778,1.155217,5.0,8.0,8.0,9.00,10.0
family,324.0,8.324074,1.237910,4.0,8.0,9.0,9.00,10.0
friends,140.0,7.564286,1.309680,5.0,7.0,7.0,8.25,10.0
solo,159.0,7.415094,1.298986,3.0,7.0,7.0,8.00,10.0
with_pet,46.0,8.021739,1.105323,6.0,7.0,8.0,9.00,10.0


## 9. Как это видит агент: примеры вызовов (выход — модели контракта)

In [12]:
from datetime import date, datetime

pd.DataFrame([d.model_dump() for d in cat.destination_options(11, "Москва", 2, [7], 7)])[
    ["name", "air_temp_c", "sea_temp_c", "travel_hours", "est_total_price", "why"]]

,name,air_temp_c,sea_temp_c,travel_hours,est_total_price,why
0,Лазаревское,16.0,16.0,5.2,80700.0,"[воздух днём ≈16°C, море ≈16°C, не купальный с..."
1,Лоо,16.0,16.0,4.1,76400.0,"[воздух днём ≈16°C, море ≈16°C, не купальный с..."
2,Дагомыс,16.0,16.0,3.8,86700.0,"[воздух днём ≈16°C, море ≈16°C, не купальный с..."
3,Центр Сочи,16.0,16.0,3.5,100700.0,"[воздух днём ≈16°C, море ≈16°C, не купальный с..."
4,Мацеста,16.0,16.0,3.3,87700.0,"[воздух днём ≈16°C, море ≈16°C, не купальный с..."
5,Хоста,16.0,16.0,3.2,73100.0,"[воздух днём ≈16°C, море ≈16°C, не купальный с..."
6,Адлер,16.0,16.0,2.8,84600.0,"[воздух днём ≈16°C, море ≈16°C, не купальный с..."
7,Имеретинская низменность (Сириус),16.0,16.0,3.0,111000.0,"[воздух днём ≈16°C, море ≈16°C, не купальный с..."
8,Красная Поляна (посёлок),10.0,NaN,3.6,93100.0,"[воздух днём ≈10°C, не купальный сезон, от аэр..."
9,Роза Хутор,10.0,NaN,3.7,117600.0,"[воздух днём ≈10°C, не купальный сезон, от аэр..."


In [13]:
offers = cat.search_hotels(date(2026, 11, 10), date(2026, 11, 17), 2, [7], pets=True, area_ids=["adler", "khosta"], limit=5)
pd.DataFrame([{"hotel": o.hotel.name, "room": o.room, "meal": o.meal, "refundable": o.refundable, "total": o.price_total} for o in offers])

,hotel,room,meal,refundable,total
0,Гостевой дом «Самшитовый двор» (Хоста),Стандарт с балконом,none,True,11036.0
1,Гостевой дом «Веранда у Елены»,Двухместный с балконом,none,True,17218.0
2,Апартаменты «Хоста-Вью»,Студия,none,False,19208.0
3,Гостевой дом «Виноградная терраса»,Семейный номер,none,True,19727.0
4,Гостевой дом «Мандариновая 21»,Двухместный номер,none,True,24378.0


In [14]:
opts = cat.transport_options("Москва", date(2026, 11, 10), "outbound", 2, [7])
pd.DataFrame([{"kind": o.kind, "carrier": o.carrier, "depart": o.depart, "arrive": o.arrive, "h": o.duration_hours, "price": o.price_total} for o in opts])

,kind,carrier,depart,arrive,h,price
0,flight,"СоколЭйр СX-565, без багажа",2026-11-10 00:40:00+03:00,2026-11-10 03:10:00+03:00,2.50,28200.0
1,flight,"СкайЮг СX-999, без багажа",2026-11-10 07:15:00+03:00,2026-11-10 09:45:00+03:00,2.50,27880.0
2,flight,Кубань Эйрвейз КЭ-620,2026-11-10 08:40:00+03:00,2026-11-10 11:10:00+03:00,2.50,29510.0
3,flight,СоколЭйр СX-114,2026-11-10 09:55:00+03:00,2026-11-10 12:25:00+03:00,2.50,30870.0
4,flight,"Кубань Эйрвейз КЭ-708, без багажа",2026-11-10 11:20:00+03:00,2026-11-10 13:50:00+03:00,2.50,19570.0
5,train,"КурортЭкспресс №137С, плацкарт",2026-11-10 11:45:00+03:00,2026-11-11 21:13:00+03:00,33.47,11490.0
6,flight,"Кубань Эйрвейз КЭ-439, без багажа",2026-11-10 12:45:00+03:00,2026-11-10 15:15:00+03:00,2.50,27160.0
7,flight,"Кубань Эйрвейз КЭ-563, без багажа",2026-11-10 14:10:00+03:00,2026-11-10 16:40:00+03:00,2.50,23480.0
8,flight,"Меридиан Авиа МА-515, без багажа",2026-11-10 15:30:00+03:00,2026-11-10 18:00:00+03:00,2.50,29410.0
9,train,"КурортЭкспресс №129С, плацкарт",2026-11-10 19:00:00+03:00,2026-11-12 04:28:00+03:00,33.47,13320.0


In [15]:
pd.DataFrame([t.model_dump() for t in cat.transfer_options("aer", "rosa_khutor", datetime(2026, 12, 20, 14, 0), 2, [7], pets=False)])

,option_id,from_place,to_place,minutes,price_total,kind,is_demo_data
0,tf__public__aer__rosa_khutor__20261220T1400,Аэропорт Сочи (Адлер),Роза Хутор,110,800.0,public,True
1,tf__shuttle__aer__rosa_khutor__20261220T1400,Аэропорт Сочи (Адлер),Роза Хутор,105,1950.0,shuttle,True
2,tf__comfort__aer__rosa_khutor__20261220T1400,Аэропорт Сочи (Адлер),Роза Хутор,70,3450.0,taxi,True
3,tf__comfort_pet__aer__rosa_khutor__20261220T1400,Аэропорт Сочи (Адлер),Роза Хутор,70,3750.0,taxi,True
4,tf__minivan__aer__rosa_khutor__20261220T1400,Аэропорт Сочи (Адлер),Роза Хутор,70,4900.0,taxi,True
5,tf__business__aer__rosa_khutor__20261220T1400,Аэропорт Сочи (Адлер),Роза Хутор,70,7000.0,taxi,True
